In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q openai python-dotenv pandas

# 2. 데이터 합성 (Synthesis)

> **📌 이 노트북의 목표**
>
> LLM(거대 언어 모델)을 활용하여 **완전히 새로운 학습 데이터를 생성**하고,
> 생성된 데이터의 품질을 **LLM이 직접 평가**하는 파이프라인을 구현한다.
>
> 이번 실습에서는 **'영화 추천'** 태스크를 위한 합성 데이터를 만들어 본다.

## 오늘 만들 파이프라인

```
   ① 프롬프트 설계        ② 합성 데이터 생성       ③ LLM 채점         ④ 선별
   ─────────────────      ─────────────────      ─────────────     ─────────
   Zero/Few/CoT           구조화 출력(JSON)        LLM as Judge      품질 필터
   역할·목표·조건 지정      일관된 형식 강제          점수 + 이유        기준 미달 제거
```

> **💡 이 흐름이 왜 중요한가?**
>
> 모델을 학습시키려면 데이터가 필요한데, 원하는 형태의 데이터가 세상에 없는 경우가 많다.
> 사람이 직접 만들면 1건당 5분씩만 잡아도 1,000건에 **83시간**이 걸린다.
>
> 그래서 요즘 AI 회사들은 위와 같은 파이프라인으로 데이터를 **만들고, 채점하고, 걸러낸다.**
> 오늘 만드는 것은 축소판이지만 **구조는 실무와 동일**하다.
> 이후 챕터에서 배울 파인튜닝의 '재료'를 준비하는 과정이라고 생각하면 된다.

**이 노트북에서 다루는 핵심 개념:**
1. **LLM 호출 함수화** — 재사용 가능한 API 호출 패턴
2. **프롬프팅 기법 비교** — Zero-shot, Few-shot, CoT의 차이를 체험
3. **구조화된 합성 데이터 생성** — JSON 스키마로 일관된 데이터 생성
4. **LLM as Judge 평가** — 생성 데이터의 품질을 LLM이 자동 평가


## 2-1. 데이터 생성을 위한 Prompt Engineering

### 2-1-1. 합성 데이터(Synthetic Data)란?

실제 수집한 데이터가 아닌, **LLM을 통해 인공적으로 만들어낸 데이터**를 의미한다.

> **💡 합성(Synthesis) vs 증강(Augmentation)의 차이**
>
> | 구분 | 데이터 합성 | 데이터 증강 |
> |------|:---:|:---:|
> | 방식 | **완전히 새로운 데이터**를 생성 | 기존 데이터를 **변형**하여 양을 늘림 |
> | 비유 | 새로운 소설을 집필 | 기존 소설의 문장 순서를 바꾸거나 단어를 교체 |
> | 예시 | "공포 영화 추천해줘" -> 새 질문-답변 쌍 생성 | 이미지 회전, 밝기 조절, 동의어 교체 |
> | 필요 시점 | 학습 데이터가 **없거나 부족**할 때 | 기존 데이터가 있지만 **양이 부족**할 때 |
>
> 3-1 챕터에서 배울 데이터 증강(RandomCrop, Flip 등)은 기존 이미지를 변형한 것이었다.
> 여기서는 **없던 데이터를 LLM으로 새로 만드는** 합성 방식을 다룬다.

**합성 데이터가 유용한 상황**

| 상황 | 설명 | 실제 예시 |
|---|---|---|
| 데이터가 아예 없음 | 새로운 서비스라 축적된 데이터가 없다 | 신규 챗봇의 초기 학습 데이터 |
| **민감 정보** | 실제 데이터를 법적으로 못 쓴다 | 의료 기록, 금융 거래, 개인정보 |
| 희귀 사례(Edge case) | 실제로는 거의 발생하지 않지만 대비해야 한다 | 사기 거래, 설비 이상, 재난 상황 |
| 불균형 해소 | 특정 클래스만 데이터가 적다 | 100건 중 긍정 95 : 부정 5 |
| 빠른 프로토타이핑 | 일단 돌아가는 것부터 만들어야 한다 | PoC 단계의 데모 |

<br>

> **⚠️ 합성 데이터의 한계도 반드시 알아야 한다**
>
> | 한계 | 설명 | 대응 |
> |---|---|---|
> | **사실성 부족** | LLM이 없는 영화, 틀린 연도를 지어낸다 (환각) | 사실이 중요한 필드는 별도 검증 |
> | **다양성 부족** | 비슷한 표현과 소재가 반복된다 | 프롬프트 변주, 시드 데이터 활용 (2-1-3 참고) |
> | **편향 증폭** | 원본 모델의 편향이 데이터에 그대로 복사된다 | 사람의 표본 검수 필수 |
> | **모델 붕괴** | AI가 만든 데이터로 AI를 반복 학습시키면 품질이 무너진다 | 실제 데이터와 **섞어서** 사용 |
>
> **모델 붕괴(Model Collapse)** 는 연구로 보고된 실제 현상이다.
> 합성 데이터만으로 학습을 반복하면, 세대를 거듭할수록 모델이
> 드문 표현을 잊고 평범한 답만 내뱉게 된다.
>
> 👉 **핵심 원칙: 합성 데이터는 실제 데이터를 "대체"하는 것이 아니라 "보충"하는 것이다.**

### 2-1-2. 프롬프트의 기본 구조

양질의 합성 데이터를 만들려면 LLM에게 **명확한 가이드라인**을 제시해야 한다.

1. **역할(Role)**: AI에게 전문성과 톤앤매너를 설정
   - 예: "당신은 세계적인 영화 평론가입니다."
2. **목표(Task)**: 수행해야 할 작업을 구체적으로 지시
   - 예: "사용자의 취향에 맞는 영화를 추천해 주세요."
3. **조건(Constraints)**: 형식, 스타일, 분량 등 제약 조건
   - 예: "영화는 반드시 한 편만 추천하고, 이유는 세 문장 이내로."

> **💡 나쁜 프롬프트 vs 좋은 프롬프트**
>
> | | 나쁜 예 | 좋은 예 |
> |---|---|---|
> | 지시 | "영화 추천해줘" | "20대 여성이 주말 저녁 혼자 볼 공포 영화를 1편 추천해줘" |
> | 형식 | (지정 없음) | "제목, 개봉연도, 추천 이유(3문장) 순서로 작성" |
> | 금지사항 | (없음) | "스포일러를 포함하지 말 것. 2000년 이전 작품 제외" |
> | 결과 | 매번 형식이 다른 긴 글 | 항상 같은 구조의 짧은 데이터 |
>
> **모호한 지시를 주면 모호한 결과가 나온다.**
> "알아서 잘"은 사람에게도 어려운 주문이다.

> **📌 프롬프트 작성 체크리스트**
>
> - [ ] **역할**을 부여했는가?
> - [ ] **무엇을** 할지 구체적으로 적었는가? (숫자, 대상, 상황)
> - [ ] **출력 형식**을 지정했는가?
> - [ ] **하지 말아야 할 것**을 명시했는가?
> - [ ] 애매한 형용사("좋은", "적절한")를 **측정 가능한 표현**으로 바꿨는가?

### 2-1-3. 합성 데이터 프롬프트의 핵심

양질의 합성 데이터를 만들기 위해서는 **다양성**과 **일관성**을 모두 확보해야 한다.

이 둘은 서로 밀고 당기는 관계다. 형식을 강하게 고정하면 다양성이 줄고, 자유롭게 두면 형식이 흐트러진다.
**형식은 고정하되 내용은 다양하게** 만드는 것이 목표다.

**1. 다양성 확보 (temperature, top_p)**

모델이 매번 새롭고 다채로운 데이터를 생성하도록 무작위성을 조절한다.

| 파라미터 | 역할 | 낮은 값 | 높은 값 |
|---------|------|--------|--------|
| `temperature` | 확률 분포의 모양 조절 | 일관적·결정적 답변 (0에 가까울수록) | 창의적·다양한 답변 (1에 가까울수록) |
| `top_p` | 후보 단어의 범위 조절 | 소수의 안전한 단어만 후보 | 더 많은 단어를 후보에 포함 |

<br>

> **💡 temperature vs top_p**
>
> `temperature`는 확률 분포의 **뾰족함/완만함**을 조절하고,
> `top_p`는 누적 확률이 일정 값에 도달할 때까지의 **후보군 범위**를 조절한다.
> 보통 둘 중 하나만 조절하고, 나머지는 기본값(1.0)으로 두는 것이 권장된다.

**두 파라미터는 LLM 전반에 통용되는 기본 개념이므로 반드시 이해하고 넘어가자.**
다만 아래 내용을 함께 알아두어야 한다.

> **⚠️ GPT-5 계열 추론 모델은 이 값들을 조절하지 않는다**
>
> 최신 추론(reasoning) 모델은 `temperature`와 `top_p`를 **기본값(1.0) 그대로** 사용한다.
> `0.3`, `0.7` 같은 다른 값을 지정해서 보내면 아래 오류가 발생한다.
>
> ```
> Unsupported parameter: 'temperature' is not supported with this model.
> ```
>
> 모델 내부에서 이미 여러 후보를 검토하는 추론 과정을 거치기 때문에,
> 겉에서 무작위성을 조절할 필요가 줄어든 것이다.
> **그래서 이번 실습 코드에는 `temperature`와 `top_p`가 등장하지 않는다.**

**1-1. 파라미터 없이 다양성을 확보하는 방법**

`temperature`를 못 쓴다면 다양성은 어떻게 만들까? **프롬프트 설계로 해결한다.**
사실 이 방법들이 파라미터 조절보다 효과가 크고, 실무에서 더 많이 쓰인다.

| 전략 | 방법 | 예시 |
|---|---|---|
| **조건 변주** | 프롬프트의 변수를 바꿔가며 반복 호출 | 장르 x 연령대 x 상황 = 조합만큼 다양해짐 |
| **시드(seed) 데이터** | 실제 데이터 몇 건을 예시로 넣고 "이런 느낌으로 다른 것" 요청 | 실제 리뷰 3건 -> 유사하지만 새로운 리뷰 생성 |
| **금지 목록** | 이미 생성된 결과를 프롬프트에 넣고 "이건 빼고" 지시 | "이미 추천한 A, B, C는 제외하고 추천" |
| **페르소나 부여** | 화자를 바꿔가며 생성 | "10대 학생처럼" / "영화 평론가처럼" |
| **관점 지정** | 같은 소재를 다른 각도로 | "장점 위주로" / "단점 위주로" |

<br>

> **💡 이번 실습에서도 이 전략을 쓴다**
>
> 아래 데이터 생성 코드에서 **장르를 4개(공포·SF·액션·로맨스)로 바꿔가며** 호출하고,
> 절반에만 **RULE(말투 규칙)** 을 적용한다.
> 파라미터를 하나도 건드리지 않고 조건 변주만으로 서로 다른 데이터를 만드는 것이다.

**1-2. 추론 강도 조절 (reasoning effort)**

그렇다면 최신 모델에서는 응답의 성격을 어떻게 조절할까?
`temperature` 대신 **`reasoning`의 `effort`** 를 사용한다.

| effort | 특징 | 적합한 작업 |
|--------|------|------------|
| `low` | 빠르고 저렴. 생각을 짧게 함 | 분류, 형식 변환, 단순 채점 |
| `medium` (기본값) | 균형 | 일반적인 생성 작업 |
| `high` | 느리고 비쌈. 오래 생각함 | 복잡한 추론, 수학, 코드 설계 |

<br>

> **💡 성격이 완전히 다른 손잡이다**
>
> - `temperature`: 답을 고를 때의 **무작위성** -> 다양성이 달라짐
> - `effort`: 답을 내기 전 **생각하는 양** -> 정확도와 비용이 달라짐
>
> effort를 높인다고 답변이 다양해지지는 않는다.
> 눈에 보이지 않는 "생각하는 토큰"도 과금되므로, **작업 난이도에 맞춰** 고르는 것이 중요하다.

**2. 일관성 확보 (구조화 출력)**

생성된 데이터가 항상 같은 구조(예: `{"movie_name": ..., "year": ...}`)를 갖도록
JSON 스키마를 지정하여 **출력 형식을 강제**한다.
이는 후속 처리(DB 저장, 모델 학습 등)를 자동화하는 데 필수적이다.

Responses API에서는 `text` 옵션에 스키마를 넣는다.
(기존 Chat Completions API의 `response_format`에 해당한다.)

> **💡 형식이 흔들리면 무슨 일이 생기나**
>
> 1,000건을 만들었는데 950건은 `{"movie_name": ...}`, 50건은 `{"title": ...}` 이라면?
> 후속 코드가 `KeyError`로 죽거나, 조용히 50건을 누락시킨다.
> **후자가 훨씬 위험하다.** 에러가 안 나서 아무도 모르기 때문이다.
> 그래서 형식은 "잘 부탁"하는 게 아니라 **스키마로 강제**해야 한다.


### 2-1-4. 프롬프팅 기법 정리

동일한 작업이라도 **프롬프트를 어떻게 설계하느냐**에 따라 LLM의 응답 품질이 크게 달라진다.
모델을 바꾸지 않고도 성능을 끌어올릴 수 있어서, 가장 비용 대비 효과가 큰 영역이다.

#### 한눈에 보기

| 기법 | 한 줄 요약 | 장점 | 단점 | 이번 실습 |
|------|-----------|------|------|:---:|
| **Zero-shot** | 예시 없이 그냥 지시 | 간단, 토큰 절약 | 형식이 들쭉날쭉 | ✅ |
| **Few-shot** | 입출력 예시 2~3개 제공 | 형식·스타일을 정확히 유도 | 프롬프트가 길어짐 | ✅ |
| **Chain-of-Thought** | "단계별로 생각해봐" | 논리적 정확성 향상 | 응답이 길어짐 | ✅ |
| **Role Prompting** | "너는 ○○ 전문가야" | 어휘·톤이 도메인에 맞춰짐 | 과신 유발 가능 | ✅ (system) |
| **Self-Consistency** | 여러 번 풀고 다수결 | 정답률 상승 | 호출 횟수만큼 비용 증가 | ❌ |
| **Self-Refine** | 스스로 비판하고 고쳐쓰기 | 품질 개선 | 2배 이상의 호출 | ❌ |
| **Prompt Chaining** | 큰 작업을 여러 단계로 분할 | 각 단계 검증 가능 | 파이프라인 복잡도 증가 | 부분 적용 |
| **ReAct** | 생각 + 도구 사용을 번갈아 | 외부 정보 활용 가능 | 구현 복잡 | ❌ (4-2 챕터) |

---

#### 1) Zero-shot — 예시 없이 직접 지시

가장 기본적인 방식이다. 그냥 하고 싶은 말을 한다.

```
공포 장르의 영화를 추천해줘.
```

- **장점**: 프롬프트가 짧아 토큰이 적게 든다. 빠르다.
- **단점**: 출력 형식을 모델이 알아서 정한다. 매번 다른 모양이 나온다.
- **언제 쓰나**: 형식이 중요하지 않은 단순 질의, 아이디어 브레인스토밍

---

#### 2) Few-shot — 예시로 패턴을 보여주기

**답을 알려주는 게 아니라, "이런 형식으로 답해"라는 본보기를 보여주는 것**이다.

```
입력: 로맨스 영화 추천해줘
출력: 영화 제목: 노팅힐 / 개봉 연도: 1999 / 추천 이유: ...

입력: 액션 영화 추천해줘
출력: 영화 제목: 매드맥스 / 개봉 연도: 2015 / 추천 이유: ...

입력: 공포 영화 추천해줘
출력:
```

- 예시 개수에 따라 **One-shot**(1개), **Few-shot**(2~5개), **Many-shot**(수십 개)이라 부른다.
- **말로 백 번 설명하는 것보다 예시 두 개가 낫다.** 사람을 가르칠 때와 똑같다.
- **언제 쓰나**: 출력 형식을 정확히 맞춰야 할 때, 특정 말투나 스타일을 흉내 내야 할 때

> **⚠️ 예시를 고를 때 주의할 점**
>
> - 예시가 **한쪽으로 치우치면** 결과도 치우친다. (예시 3개가 전부 90년대 영화면 90년대만 추천한다)
> - 예시에 **오류가 있으면** 그 오류까지 따라 한다.
> - 예시 개수를 늘린다고 계속 좋아지지 않는다. 보통 **2~5개**에서 효과가 포화된다.

---

#### 3) Chain-of-Thought (CoT) — 생각 과정을 밖으로 꺼내기

LLM이 복잡한 문제를 풀 때 **중간 추론 과정을 명시적으로 생성**하도록 유도하는 기법이다.

```
공포 영화를 추천해줘.
단계별로 생각해봐:
1) 공포 장르의 핵심 요소는 무엇인가?
2) 그 요소를 잘 살린 작품은?
3) 그중 가장 대중적으로 인정받은 작품은?
```

- 대표적인 마법의 문장: **"Let's think step by step"** (단계별로 생각해보자)
- **왜 효과가 있나?** 사람도 암산으로 어려운 문제를 종이에 적으면서 풀면 잘 푼다.
  중간 과정을 텍스트로 뱉으면, 그 텍스트가 다시 다음 추론의 근거가 되기 때문이다.
- **언제 쓰나**: 수학, 논리 추론, 여러 조건을 동시에 따져야 하는 판단

> **💡 두 가지 변형**
>
> | 종류 | 방법 |
> |---|---|
> | **Zero-shot CoT** | 그냥 "단계별로 생각해봐" 한 줄 추가 |
> | **Few-shot CoT** | 예시 자체에 풀이 과정을 담아서 제공 |
>
> Few-shot CoT가 더 강력하지만 프롬프트가 훨씬 길어진다.

> **⚠️ 추론 모델 시대에는 CoT의 위상이 달라졌다**
>
> 우리가 쓰는 `gpt-5-nano` 같은 **추론(reasoning) 모델은 시키지 않아도 속으로 단계를 밟는다.**
> 그래서 "단계별로 생각해봐"를 붙여도 예전만큼 극적인 향상이 나타나지 않는다.
>
> 그럼 안 배워도 될까? **아니다.**
> - 모든 모델이 추론 모델은 아니다. 소형 모델·오픈소스 모델에서는 여전히 효과가 크다.
> - **추론 과정을 사람에게 보여줘야 할 때**는 여전히 CoT를 명시적으로 요청한다.
>   (모델 내부의 추론은 사용자에게 노출되지 않는다)

---

#### 4) Role Prompting — 역할 부여

**"당신은 20년 경력의 영화 평론가입니다."**

모델에게 역할을 주면 그 역할에 맞는 어휘와 관점, 상세도로 답한다.
이번 실습의 `system_prompt`가 정확히 이 기법이다.

- **효과**: 도메인 용어 사용, 답변 깊이, 톤앤매너가 달라진다
- **⚠️ 주의**: 역할을 준다고 **없던 지식이 생기지는 않는다.**
  "당신은 의사입니다"라고 해도 의학적으로 틀린 답을 자신 있게 할 수 있다.
  오히려 **자신감만 올라가서 더 위험**해질 수 있다.

---

#### 5) Self-Consistency — 여러 번 풀고 다수결

같은 질문을 **여러 번** 물어보고, 가장 많이 나온 답을 채택한다.

```
질문 -> 답변 A (3회)
     -> 답변 B (1회)   =>  A 채택
     -> 답변 C (1회)
```

- 정답이 하나로 정해진 문제(수학, 분류)에서 정확도가 눈에 띄게 올라간다.
- **단점**: 5번 호출하면 비용도 5배다.
- **⚠️ 한계**: 매번 다른 답이 나와야 다수결이 의미가 있는데,
  `temperature`를 조절할 수 없는 모델에서는 효과가 제한적이다.

---

#### 6) Self-Refine — 스스로 비판하고 고쳐쓰기

**생성 -> 자기 비판 -> 수정**을 반복한다.

```
1차: "공포 영화를 추천해줘"                    -> 초안 생성
2차: "위 답변의 문제점을 3가지 지적해줘"        -> 자기 비판
3차: "지적한 내용을 반영해서 다시 써줘"          -> 개선본
```

- 글쓰기, 코드 작성처럼 **정답이 하나가 아닌 작업**에서 효과적이다.
- 오늘 배울 **LLM as Judge와 발상이 같다.** 채점자를 자기 자신으로 두는 셈이다.
- **⚠️ 한계**: 자기가 못 보는 오류는 비판도 못 한다. 무한히 좋아지지 않는다.

---

#### 7) Prompt Chaining — 작업 쪼개기

큰 작업을 한 번에 시키지 않고 **여러 단계로 나눠서** 순차 호출한다.

```
[1단계] 영화 5편 후보 뽑기
   ↓
[2단계] 각 후보의 추천 이유 작성
   ↓
[3단계] JSON 형식으로 정리
```

- **장점**: 각 단계의 결과를 **검증하고 고칠 수 있다.** 어디서 틀렸는지 추적 가능하다.
- **단점**: 호출 횟수가 늘고 파이프라인이 복잡해진다.
- 이번 실습의 **생성 -> 평가** 구조도 일종의 체이닝이다.

---

#### 8) ReAct — 생각과 행동을 번갈아

**Reasoning(추론) + Acting(행동)** 의 합성어다.
모델이 "생각 -> 도구 사용 -> 결과 관찰 -> 다시 생각"을 반복한다.

```
생각: 최신 개봉작을 알아야겠다
행동: 웹 검색 호출
관찰: 검색 결과 수신
생각: 이 중 공포 장르는...
```

- LLM이 모르는 **최신 정보나 외부 데이터**가 필요할 때 쓴다.
- **4-2 챕터의 AI Agent가 바로 이 구조**다. 지금은 이름만 알아두자.

---

#### 🎯 어떤 기법을 골라야 하나?

```
   출력 형식을 정확히 맞춰야 한다        ->  Few-shot + 구조화 출력
   복잡한 판단·계산이 필요하다           ->  CoT (+ 추론 모델이면 effort 상향)
   전문 분야의 답이 필요하다             ->  Role Prompting
   정답이 하나인데 정확도가 중요하다      ->  Self-Consistency
   글의 완성도를 높여야 한다             ->  Self-Refine
   단계가 많고 중간 검증이 필요하다       ->  Prompt Chaining
   외부 정보·도구가 필요하다             ->  ReAct (4-2 챕터)
```

> **💡 기법은 조합해서 쓴다**
>
> 실무에서 하나만 쓰는 경우는 드물다. 이번 실습만 해도
> **Role Prompting**(시스템 프롬프트) + **Few-shot**(형식 예시) + **구조화 출력**(JSON 스키마)
> 을 함께 쓴다.
>
> 그리고 **모든 기법에는 비용이 따른다.** 프롬프트가 길어지거나, 호출이 늘거나, 응답이 길어진다.
> **공짜는 없다. 작업에 필요한 만큼만 쓴다.**

아래 코드에서 Zero-shot / Few-shot / CoT **세 가지의 실제 출력 차이**를 직접 확인해 보자.


## 2-2. 실습 코드

### 2-2-1. 환경 설정

2_1_Settings에서 설정한 것과 동일한 방식으로 API 키를 로드한다.
`.env` 파일에 `GMS_KEY`가 저장되어 있어야 한다.


In [ ]:
import json
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
from os import getenv
from openai import OpenAI

# .env 파일에서 API 키 로드
load_dotenv()
GMS_KEY = getenv('GMS_KEY')

if GMS_KEY:
    print('API 키 로드 성공!')
else:
    print('ERROR: .env 파일에 GMS_KEY가 설정되지 않았습니다.')


### 2-2-2. LLM 호출 함수 구현

API 호출을 매번 직접 작성하면 코드가 반복된다.
**재사용 가능한 함수**로 만들어 두면 이후 모든 호출에서 편리하게 사용할 수 있다.

> **💡 왜 함수로 만드는가?**
>
> 이 노트북에서만 해도 LLM을 **데이터 생성, 프롬프팅 비교, 품질 평가** 등 여러 번 호출한다.
> 매번 `client.responses.create(...)`를 작성하는 대신,
> `chat_completion(prompt)` 한 줄로 호출할 수 있도록 함수화한다.


In [ ]:
# ========== GMS API 클라이언트 생성 ==========
client = OpenAI(
    api_key=GMS_KEY,
    base_url='https://gms.ssafy.io/gmsapi/api.openai.com/v1/',
)


# ========== 재사용 가능한 LLM 호출 함수 ==========
# 왜 함수로 만드는가?
# → 이 실습에서 LLM을 여러 번 호출한다 (데이터 생성, 평가 등).
#   매번 client.responses.create(...)를 작성하면 코드가 반복된다.
#   함수로 만들면 prompt만 바꿔가며 재사용할 수 있다.
#
# 파라미터 설명
#   prompt          : 사용자 메시지
#   system_prompt   : 역할/규칙을 지정하는 지시문 (Responses API의 instructions)
#   model           : 사용할 모델명
#   effort          : 추론 강도 ('low' | 'medium' | 'high')
#                     temperature 대신 사용하는 조절 손잡이다
#   text_format     : 구조화 출력 스키마 (선택)
def chat_completion(prompt, system_prompt=None, model='gpt-5-nano',
                    effort='medium', text_format=None):
    """LLM을 호출하여 응답 텍스트를 반환한다."""
    kwargs = {
        'model': model,
        'input': prompt,                  # Responses API는 messages 대신 input
        'reasoning': {'effort': effort},  # temperature 자리를 대신하는 파라미터
    }

    # 시스템 프롬프트는 instructions로 전달한다.
    # Chat Completions에서 messages에 role='system'을 넣던 것과 같은 역할이다.
    if system_prompt:
        kwargs['instructions'] = system_prompt

    if text_format:
        kwargs['text'] = text_format

    response = client.responses.create(**kwargs)

    # output_text: 생성된 최종 텍스트를 바로 꺼내주는 속성
    return response.output_text


# 테스트
test = chat_completion('안녕? 간단한 인사말을 해줘.', effort='low')
print('테스트 응답:', test)


### 2-2-3. 프롬프팅 기법 비교 (Zero-shot / Few-shot / CoT)

동일한 작업(영화 추천)에 대해 세 가지 기법을 적용하고, 결과 차이를 직접 확인한다.

**공정한 비교를 위한 통제**

- 시스템 프롬프트: 세 기법 **모두 동일**
- 사용자 질문: 세 기법 **모두 동일**
- 바뀌는 것: **사용자 프롬프트를 어떻게 구성하는가** 하나뿐

> **💡 관찰 포인트**
>
> | 무엇을 볼까 | 확인할 것 |
> |---|---|
> | **형식의 일관성** | Few-shot 결과가 제공한 예시의 형식(제목/연도/이유 순서)을 따르는가? |
> | **추론의 깊이** | CoT 결과에 "왜 그 영화인가"에 대한 단계적 근거가 보이는가? |
> | **응답 길이** | 세 결과의 길이 차이는? 길다고 항상 좋은가? |
> | **어휘와 톤** | 시스템 프롬프트로 부여한 '역할'이 세 결과 모두에 반영되었는가? |

<br>

> **⚠️ 미리 알아두기**
>
> 우리가 쓰는 모델은 **추론 모델**이라, 시키지 않아도 속으로 단계를 밟는다.
> 그래서 **CoT의 효과가 교과서만큼 극적이지 않을 수 있다.**
>
> 차이가 작게 나오더라도 실패가 아니다.
> "최신 모델에서는 CoT가 어느 정도 내장되어 있다"는 것을 **직접 확인한 결과**로 받아들이자.
> 대신 **Few-shot의 형식 통제 효과는 뚜렷하게** 나타날 것이다.


In [ ]:
# 공통 시스템 프롬프트 (세 기법 모두 동일한 역할 부여)
SYSTEM_PROMPT = '''당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 \'시네마스터\'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다.
추천할 때는 반드시 영화 제목, 개봉 연도, 그리고 추천 이유를 포함해야 합니다.'''

user_query = '공포 영화를 추천해줘'

In [ ]:
# ===== 1. Zero-shot: 예시 없이 직접 지시 =====
zero_shot_prompt = user_query  # 아무 예시도 없이 그냥 질문

print('=' * 50)
print('1. Zero-shot 결과:')
print('=' * 50)
print(chat_completion(zero_shot_prompt, SYSTEM_PROMPT))


In [ ]:
# ===== 2. Few-shot: 입출력 예시 2개 제공 =====
# 왜 예시를 주는가?
# → LLM에게 "이런 형식으로 대답해"라는 패턴을 보여주는 것
#   예시가 있으면 형식과 톤앤매너를 더 정확히 맞출 수 있다
few_shot_prompt = f'''다음은 영화 추천 예시입니다:

질문: 로맨스 영화를 추천해줘
답변: 영화 제목: 노트북 (The Notebook)
개봉 연도: 2004년
추천 이유: 시대를 초월한 순수한 사랑 이야기로, 감동적인 클래식 로맨스 영화입니다.

질문: 코미디 영화를 추천해줘
답변: 영화 제목: 행오버 (The Hangover)
개봉 연도: 2009년
추천 이유: 예측 불가능한 전개와 유쾌한 유머가 가득한 코미디 영화입니다.

질문: {user_query}
답변:'''

print('\n' + '=' * 50)
print('2. Few-shot 결과:')
print('=' * 50)
print(chat_completion(few_shot_prompt, SYSTEM_PROMPT))

In [ ]:
# ===== 3. Chain-of-Thought: 단계별 추론 유도 =====
# 왜 단계를 나누는가?
# → 복잡한 작업에서 LLM이 "생각하는 과정"을 거치면 더 정확한 결과를 낸다
cot_prompt = f'''{user_query}

영화를 추천하기 전에 다음 단계를 따라 생각해주세요:
1단계: 공포 장르의 핵심 요소가 무엇인지 정의합니다
2단계: 이 요소들을 잘 갖춘 대표적인 공포 영화들을 떠올립니다
3단계: 그 중 가장 추천할 만한 영화 1개를 선택하고 이유를 설명합니다

위 단계를 따라 추론 과정을 보여주고, 최종 추천을 해주세요.'''

print('\n' + '=' * 50)
print('3. Chain-of-Thought 결과:')
print('=' * 50)
print(chat_completion(cot_prompt, SYSTEM_PROMPT))

### 2-2-4. JSON 파싱 방법 2가지

LLM의 응답을 구조화된 데이터로 받는 방법은 두 가지가 있다.

| 방법 | 설명 | 장점 | 단점 |
|------|------|------|------|
| `text` 옵션 (API 레벨) | API 파라미터로 JSON 스키마 강제 | 파싱 실패 없음, 정확함 | 일부 API만 지원 |
| **프롬프트에 JSON 형식 지정** + 수동 파싱 | 프롬프트에 "JSON으로 답해"라고 지시 | 어떤 API든 사용 가능 | 파싱 실패 가능성 있음 |

이 노트북에서는 `text` 옵션(방법 1)을 주로 사용하되,
범용적으로 사용할 수 있는 `json_parsing()` 유틸리티 함수(방법 2)도 함께 구현해 둔다.

> **💡 방법 2를 왜 배워두는가?**
>
> 스키마 강제는 편하지만 **모든 API가 지원하지는 않는다.**
> 오픈소스 모델을 직접 서빙하거나, 구형 API를 쓰거나,
> LangChain 같은 프레임워크를 거칠 때는 방법 2가 필요하다.
> **실무에서 방법 2를 쓸 일이 생각보다 많다.**

> **⚠️ 방법 2에서 파싱이 실패하는 대표 유형**
>
> | 유형 | 실제 응답 예시 | 대응 |
> |---|---|---|
> | 코드블록으로 감쌈 | ` ```json { ... } ``` ` | 백틱 제거 후 파싱 (아래 함수가 처리) |
> | 앞뒤 설명 추가 | `물론이죠! { ... } 도움이 되었길` | 정규식으로 `{...}` 구간만 추출 |
> | 따옴표 오류 | `{'name': '값'}` (작은따옴표) | JSON은 큰따옴표만 허용 -> 재시도 |
> | 마지막 쉼표 | `{"a": 1, "b": 2,}` | 후처리로 제거 |
> | 중간에 잘림 | `{"a": 1, "b":` | 출력 토큰 한도 초과 -> 한도 상향 |
>
> **실무 원칙: 파싱은 반드시 `try / except`로 감싸고, 실패한 건은 버리거나 재시도한다.**
> 1,000건 중 3건이 실패했다고 전체 파이프라인이 멈추면 안 된다.


In [ ]:
# ========== JSON 파싱 유틸리티 함수 ==========
# LLM이 ```json ... ``` 형태로 응답할 때, 그 안의 JSON만 추출하는 함수
# 왜 필요한가?
# → response_format을 지원하지 않는 API도 있고,
#   프롬프트로 JSON을 요청하면 앞뒤에 설명 텍스트가 붙는 경우가 많다.
#   이 함수는 그 중에서 JSON 부분만 깔끔하게 추출한다.
def json_parsing(output_text):
    """LLM 응답에서 JSON 부분을 추출하여 딕셔너리로 변환한다. 실패하면 None을 반환한다."""
    try:
        # ```json ... ``` 블록이 있으면 그 안의 내용만 추출
        if '```json' in output_text:
            output_text = output_text[output_text.index('```json') + len('```json'):].strip()
            output_text = output_text[:output_text.index('```')].strip()
        return json.loads(output_text)
    except (json.JSONDecodeError, ValueError) as e:
        print(f'JSON 파싱 오류: {e}')
        return None

print('json_parsing 함수 정의 완료')


### 2-2-5. 구조화된 합성 데이터 생성

프롬프트 엔지니어링의 3요소(역할, 목표, 조건)를 적용하여 영화 추천 데이터를 생성한다.
여기서는 `text` 옵션(API 레벨 강제)을 사용한다.

**이번 생성에 적용된 기법**

| 기법 | 어디에 적용되었나 |
|---|---|
| Role Prompting | 시스템 프롬프트의 "영화 평론가" 역할 부여 |
| 조건 변주 | 장르를 4개로 바꿔가며 반복 호출 (2-1-3의 다양성 전략) |
| 구조화 출력 | JSON 스키마로 5개 필드 강제 |
| effort 조절 | 창작이 섞인 작업이므로 `medium` 사용 |

<br>

> **💡 의도적으로 규칙을 지키기 어려운 데이터도 함께 생성한다**
>
> 모든 데이터가 완벽하면 이후 평가 단계에서 점수 변별력이 없어진다.
> 아래에서 **RULE이 없는 "일반 생성"과 RULE이 있는 "규칙 생성"** 을 모두 만들어서
> 평가 시 점수 차이가 나타나도록 구성한다.
>
> ```
>   공포   (RULE 적용)   ┐
>   SF     (RULE 적용)   ┘ -> 친근하고 호들갑스러운 말투
>   액션   (RULE 미적용) ┐
>   로맨스 (RULE 미적용) ┘ -> 기본 말투
> ```
>
> **왜 일부러 섞는가?**
> 전부 5점이 나오면 그게 데이터가 좋아서인지 채점기가 후한 건지 구분할 수 없다.
> 규칙을 안 지킨 데이터를 섞어두면, 점수가 갈리는지 확인해서
> **채점기가 제대로 작동하는지 검증**할 수 있다.
> 실무에서 평가 시스템을 만들 때 반드시 하는 작업이다.

> **📌 결과가 나오면 직접 비교해 보자**
>
> 출력된 네 건에서 **`recommended_reason` 항목만** 나란히 읽어보자.
> 공포·SF와 액션·로맨스의 **말투가 다른 것**이 보이는가?
> 프롬프트에 문장 하나를 추가한 것만으로 생긴 차이다.


In [ ]:
# ========== 1. 프롬프트 설계 ==========

# 시스템 프롬프트: 역할(Role) + 목표(Task)
GEN_SYSTEM_PROMPT = '''당신은 세상의 모든 영화를 꿰뚫고 있는 영화 전문가 \'시네마스터\'입니다.
사용자의 요청에 맞춰 영화를 추천하는 역할을 맡고 있습니다. 영화는 반드시 하나만 추천합니다.'''

# 추가 규칙: 조건(Constraints) — 말투/스타일 통제
RULE = '''친구가 소개해주는 듯 부드럽고 친근한 말투로 답변합니다.
특히, recommended_reason 항목에서는 친구가 엄청 호들갑 떨듯이 설명해 주세요.'''

# ========== 2. JSON 응답 형식 (API 레벨 강제) ==========
# [주의] strict 모드에는 두 가지 필수 조건이 있다.
#   1) properties의 모든 키가 required에 들어가야 한다
#   2) additionalProperties를 반드시 False로 지정해야 한다
# 하나라도 빠지면 400 에러가 발생한다.
# 또한 name에는 영문/숫자/언더바만 사용할 수 있다 (한글 불가).
movie_text_format = {
    'format': {
        'type': 'json_schema',
        'name': 'movie_recommendation',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'movie_name': {'type': 'string'},
                'year': {'type': 'integer'},
                'reason': {'type': 'string'},
                'description': {'type': 'string', 'description': '영화에 대한 설명'},
                'recommended_reason': {'type': 'string', 'description': '추천 추가 이유'},
            },
            'required': ['movie_name', 'year', 'reason', 'description', 'recommended_reason'],
            'additionalProperties': False,
        },
    },
}

# ========== 3. 합성 데이터 생성 ==========
# 점수 다양성을 확보하기 위해 두 가지 조건으로 생성한다:
# - RULE 적용 (친근한 말투 규칙을 따르는 데이터)
# - RULE 미적용 (규칙 없이 기본 톤으로 생성한 데이터)
# → 이후 평가에서 RULE 준수 여부를 기준으로 평가하면 점수 차이가 발생한다.

generation_configs = [
    {'genre': '공포', 'use_rule': True,  'label': '공포 (RULE 적용)'},
    {'genre': 'SF',   'use_rule': True,  'label': 'SF (RULE 적용)'},
    {'genre': '액션', 'use_rule': False, 'label': '액션 (RULE 미적용)'},
    {'genre': '로맨스', 'use_rule': False, 'label': '로맨스 (RULE 미적용)'},
]

synthetic_data = []

for config in generation_configs:
    genre = config['genre']
    label = config['label']
    # RULE 적용 여부에 따라 시스템 프롬프트를 다르게 구성
    if config['use_rule']:
        sys_prompt = GEN_SYSTEM_PROMPT + '\n' + RULE
    else:
        sys_prompt = GEN_SYSTEM_PROMPT  # RULE 없이 기본 톤

    print(f'\n{label} 생성 중...')
    output = chat_completion(
        prompt=f'{genre} 영화를 추천해줘',
        system_prompt=sys_prompt,
        effort='medium',   # 창작이 섞인 생성 작업이므로 기본 강도를 사용
        text_format=movie_text_format,
    )
    data = json.loads(output)
    data['_rule_applied'] = config['use_rule']  # 메타 정보: 평가 시 참고용
    data['_genre'] = genre
    synthetic_data.append(data)
    pprint(data)

print(f'\n생성 완료: 총 {len(synthetic_data)}건 (RULE 적용 {sum(c["use_rule"] for c in generation_configs)}건, 미적용 {sum(not c["use_rule"] for c in generation_configs)}건)')


## 2-3. 생성 데이터 평가 (LLM as a Judge)

### 2-3-1. LLM as a Judge란?

사람이 수많은 합성 데이터의 품질을 하나하나 검수하는 것은 비효율적이다.
**LLM as a Judge**는 이 검수 과정을 자동화하기 위해 **다른 LLM을 '평가자'로 활용**하는 기법이다.

> **💡 핵심 장점**
>
> 단순히 "좋다/나쁘다" 판정을 넘어,
> **"왜 그렇게 평가했는지" 이유까지 생성** 하여 데이터의 어떤 부분을 개선해야 할지
> 구체적인 피드백을 얻을 수 있다.

**왜 이 기법이 필요한가?**

| 평가 방법 | 1,000건 검수 비용 | 품질 | 한계 |
|---|---|---|---|
| 사람이 전수 검사 | 며칠 ~ 몇 주 | 가장 정확 | 느리고 비쌈, 확장 불가 |
| 규칙 기반 (길이, 키워드) | 즉시 | 낮음 | "말투가 친근한가" 같은 판단 불가 |
| **LLM as Judge** | 수 분 | 준수 | 편향 존재 (아래 참고) |

---

### 2-3-2. 평가 방식의 종류

평가에는 크게 세 가지 방식이 있다. 목적에 따라 골라 쓴다.

| 방식 | 설명 | 언제 쓰나 |
|---|---|---|
| **Pointwise (점수형)** | 결과 하나에 점수를 매김 (1~5점) | 대량 데이터 품질 필터링 ← **이번 실습** |
| **Pairwise (비교형)** | A와 B 중 어느 쪽이 나은지 고름 | 모델 A vs 모델 B 성능 비교 |
| **Reference-based (정답 대조)** | 모범 답안과 비교 | 정답이 명확한 태스크 |

<br>

> **💡 Pairwise가 더 정확한 경우가 많다**
>
> 사람도 "이 글은 몇 점인가?"보다 "둘 중 어느 게 나은가?"를 훨씬 잘 판단한다.
> LLM도 마찬가지다. 다만 비교 대상이 필요하고, 1,000건을 서로 비교하려면
> 호출 횟수가 폭증하기 때문에 **대량 필터링에는 Pointwise를 쓴다.**

---

### 2-3-3. 평가 설계의 핵심 3가지

1. **평가 기준 설정**
   - 평가 목적을 명확히 해야 한다
   - 이번 실습: 영화 정보의 사실 여부가 아니라, 우리가 지시한 **RULE(친근한 말투, 호들갑 떠는 설명)을 얼마나 잘 이행했는지** 를 평가
   - ⚠️ **"좋은 답변인가?"는 나쁜 기준이다.** '좋다'의 정의가 사람마다 다르기 때문이다.
     **"3문장 이내인가", "친근한 말투인가"** 처럼 판단 가능한 형태로 쪼개야 한다.

2. **일관성 확보**
   - 평가자 LLM은 창의적일 필요가 없다
   - 동일한 입력에 대해 항상 **일관되고 객관적인 평가** 를 내려야 한다
   - 예전에는 `temperature=0`으로 설정하여 **결정적(deterministic)** 응답을 유도했다
   - ⚠️ 하지만 GPT-5 계열 추론 모델은 `temperature`를 지원하지 않는다.
     대신 **채점 기준을 프롬프트에 아주 구체적으로 못 박는 것** 이 일관성 확보의 핵심이 된다.
     (아래 `JUDGE_SYSTEM_PROMPT`에서 1~5점의 의미를 명시한 이유가 이것이다)
   - 이번 실습에서는 채점이 단순 판정 작업이므로 `effort='low'`를 사용한다.
     빠르고 저렴하면서도 기준이 명확해 판정 품질에는 큰 영향이 없다.
   - 그럼에도 같은 데이터를 다시 평가하면 점수가 1점 정도 흔들릴 수 있다.
     **점수의 절대값보다 그룹 간 상대 차이** 에 주목하자.

3. **체계적인 평가 프롬프트**
   - 평가자에게 **[원래 지시사항], [생성 결과], [평가 기준]** 을 모두 명시적으로 전달
   - 출력도 `score`(점수)와 `comment`(이유)를 포함하는 **JSON 형식** 으로 강제
   - 점수 구간의 의미를 **숫자마다 정의** 한다 (5점=완전 충족, 3점=부분 충족 ...)

---

### 2-3-4. ⚠️ LLM as Judge의 한계 — 반드시 알아야 할 편향

LLM 채점을 **맹신하면 안 된다.** 연구를 통해 알려진 대표적인 편향들이다.

| 편향 | 내용 | 대응 |
|---|---|---|
| **자기 편향** (Self-bias) | 자기가(또는 같은 계열 모델이) 만든 답에 후한 점수를 준다 | 생성 모델과 **다른 모델**로 채점 |
| **위치 편향** (Position bias) | 먼저 제시된 답을 더 높게 평가한다 | 순서를 바꿔 두 번 평가 후 평균 |
| **길이 편향** (Verbosity bias) | 길고 자세한 답을 더 좋게 본다 | 기준에 "간결함"을 명시 |
| **관대함 편향** | 전반적으로 후한 점수를 준다 (3점 이하를 잘 안 줌) | 점수 정의를 엄격히, 낮은 점수 예시 제공 |

<br>

> **💡 그럼 어떻게 써야 하나 — "1차 필터"로 쓴다**
>
> ```
>   합성 데이터 1,000건
>        ↓  LLM as Judge (자동)
>   명백히 나쁜 것 제거 -> 800건 통과
>        ↓  사람이 표본 검수 (50건만)
>   최종 품질 확인
> ```
>
> **완전 자동화가 아니라, 사람이 봐야 할 양을 줄이는 도구**로 쓴다.
> 1,000건을 다 보던 것을 50건만 보면 되니, 그것만으로도 충분히 큰 이득이다.
>
> 👉 AI 활용의 일반 원칙이기도 하다.
> **사람을 대체하는 것이 아니라, 사람이 집중할 지점을 좁혀주는 것.**

<br>

> **📌 채점기를 믿어도 되는지 확인하는 방법**
>
> 1. **정답을 아는 데이터를 섞는다** ← 이번 실습에서 하는 것
>    규칙을 지킨 것과 안 지킨 것을 섞어 점수가 갈리는지 본다
> 2. **사람 평가와 비교한다**
>    50건 정도를 사람이 채점해서 LLM 점수와 얼마나 일치하는지 확인한다
> 3. **같은 데이터를 여러 번 평가한다**
>    점수가 크게 흔들리면 기준이 모호하다는 뜻이다


### 2-3-5. 평가 실습 코드

아래에서 평가 함수를 정의하고, 먼저 단일 데이터에 대해 테스트한 뒤,
모든 합성 데이터에 대해 다중 기준으로 평가하는 과정까지 진행한다.

> **💡 평가 프롬프트에서 눈여겨볼 것**
>
> `JUDGE_SYSTEM_PROMPT`를 읽어보자. 이 프롬프트가 앞서 배운 원칙을 어떻게 구현했는지 보인다.
>
> - **역할 부여**: 평가자라는 정체성을 준다 (Role Prompting)
> - **점수 정의**: 1~5점 각각의 의미를 문장으로 못 박는다 (일관성 확보)
> - **출력 형식**: score와 comment를 JSON으로 강제 (구조화 출력)
>
> `temperature=0`을 못 쓰는 대신, **프롬프트가 그 역할을 대신하고 있다.**


In [ ]:
# ========== 1. 평가자 시스템 프롬프트 ==========
JUDGE_SYSTEM_PROMPT = '''당신의 역할은 모델 답변 자동 평가자입니다.

1. 입력 형식
    - 입력 프롬프트: [instruction]
    - 모델 답변: [output]
    - 평가 기준: [criteria]

2. 작업 지시
    - [instruction]에 따른 모델 결과물인 [output]을 평가합니다.
    - [output]은 [criteria]를 충족하는지 평가합니다.

3. 채점 원칙 (1–5점, 정수만)
    - 5점 (탁월): 기준을 완전히 충족.
    - 4점 (우수): 대체로 충족. 사소한 흠만 있음.
    - 3점 (보통): 핵심은 맞지만 약점 존재.
    - 2점 (미흡): 중요한 요구를 놓침.
    - 1점 (부적합): 전반적으로 요청과 어긋남.

4. 출력 형식 (엄격 준수)
    - "score"는 1–5점 정수.
    - "comment"는 한국어 1–3문장.
    - 지정된 JSON 형식을 준수.'''

# ========== 2. 평가 JSON 응답 형식 ==========
# 생성용 스키마와 동일하게 strict 모드 조건(required 전체 포함 + additionalProperties=False)을 지킨다.
judge_text_format = {
    'format': {
        'type': 'json_schema',
        'name': 'evaluation_result',
        'strict': True,
        'schema': {
            'type': 'object',
            'properties': {
                'score': {'type': 'integer'},
                'comment': {'type': 'string', 'description': '평가 이유'},
            },
            'required': ['score', 'comment'],
            'additionalProperties': False,
        },
    },
}

# ========== 3. 평가 함수 ==========
# 여러 데이터 × 여러 기준으로 반복 평가해야 하므로 함수로 만든다.
# 반환값은 {'score': 1~5, 'comment': '평가 이유'} 형태의 딕셔너리이다.
#   instruction : 원본 지시문 (데이터 생성 시 사용한 프롬프트)
#   output      : 평가할 생성 결과
#   criteria    : 평가 기준 문자열
def evaluate_with_llm(instruction, output, criteria):
    """LLM as Judge: 생성 결과를 평가하여 점수와 코멘트를 반환한다."""
    user_prompt = f'''입력 프롬프트: {instruction}
모델 답변: {output}
평가 기준: {criteria}'''

    result_text = chat_completion(
        prompt=user_prompt,
        system_prompt=JUDGE_SYSTEM_PROMPT,
        effort='low',   # 채점은 기준이 명확한 단순 판정이므로 낮은 강도로 충분
                        # (예전 코드의 temperature=0 자리를 대신한다)
        text_format=judge_text_format,
    )
    return json.loads(result_text)


# ========== 4. 단일 평가 테스트 ==========
test_data = synthetic_data[0]
print('평가 대상:')
pprint(test_data)

result = evaluate_with_llm(
    instruction=GEN_SYSTEM_PROMPT,
    output=json.dumps(test_data, ensure_ascii=False),
    criteria='요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보가 모두 포함되었는지',
)
print(f'\n점수: {result["score"]} / 5')
print(f'평가: {result["comment"]}')


### 2-3-6. 다중 기준 평가 + 결과 정리

실제 품질 평가에서는 **하나의 기준이 아닌 여러 기준** 으로 평가해야 한다.
모든 합성 데이터에 대해 여러 기준으로 평가하고, 결과를 표로 정리해 보자.

**왜 여러 기준으로 나누는가?**

"이 데이터 좋아?"라고 물으면 답이 뭉뚱그려진다.
기준을 쪼개면 **어느 부분이 문제인지** 알 수 있다.

| 기준 | 무엇을 보나 | 낮으면 무엇을 고쳐야 하나 |
|---|---|---|
| **충실도** | 요청한 내용에 맞게 답했는가 | 목표(Task) 지시를 더 명확히 |
| **구체성** | 두루뭉술하지 않고 근거가 있는가 | 조건(Constraints)에 분량·항목 추가 |
| **말투 규칙** | 지정한 스타일을 지켰는가 | RULE 문장을 더 강하게 |

<br>

> **📌 좋은 평가 기준을 만드는 요령**
>
> - **하나의 기준은 한 가지만 묻는다.** "정확하고 친절한가"는 두 개를 섞은 나쁜 기준이다.
> - **판단 가능한 말로 쓴다.** "좋은가" -> "3문장 이내인가", "예시가 있는가"
> - **기준끼리 겹치지 않게** 한다. 겹치면 같은 문제로 두 번 감점된다.

> **💡 왜 RULE 적용/미적용 데이터를 섞었는가?**
>
> 모든 데이터가 동일한 조건으로 생성되면 평가 점수가 전부 비슷하게 나와서
> **점수의 변별력** 을 확인하기 어렵다.
> RULE(친근한 말투 규칙)이 적용된 데이터와 그렇지 않은 데이터를 함께 평가하면,
> "말투 규칙 준수" 기준에서 **점수 차이가 나타나는 것** 을 관찰할 수 있다.
> 이것이 LLM as Judge의 **변별력을 검증** 하는 방법이다.

> **📊 결과에서 확인할 것**
>
> 1. **말투규칙 열**에서 RULE 적용 그룹의 점수가 더 높은가?
> 2. **충실도·구체성**은 두 그룹이 비슷한가?
>    (RULE은 말투에 대한 규칙이므로, 다른 기준까지 크게 차이 나면 기준이 겹친 것이다)
> 3. 점수가 예상과 다르다면 -> **실패가 아니라 관찰 결과다.**
>    `temperature`를 고정할 수 없어 흔들린 것일 수 있다.
>    실무에서는 이래서 **여러 번 평가해 평균** 을 낸다.


In [ ]:
# ========== 다중 평가 기준 정의 ==========
# 점수 다양성을 위해 "쉬운 기준"과 "까다로운 기준"을 함께 배치한다.
# - 기준 1~2: 대부분의 데이터가 충족하기 쉬운 기본 기준
# - 기준 3: RULE을 적용한 데이터만 높은 점수를 받을 수 있는 까다로운 기준
criteria_list = [
    '요청의 충실도: 요청된 장르에 맞는 영화를 추천했는지, 필수 정보(영화명, 연도, 이유)가 모두 포함되었는지 평가',
    '추천 이유의 구체성: 추천 이유가 해당 장르의 특성과 연결되어 구체적이고 설득력 있는지, 단순히 "재미있다" 수준의 피상적 설명은 3점 이하로 평가',
    '말투 규칙 준수: 친구가 소개해주는 듯 부드럽고 친근한 말투인지, 특히 추천 이유에서 호들갑 떠는 듯한 열정적 표현이 사용되었는지 평가. 격식체나 딱딱한 문어체는 2점 이하로 평가',
]

# ========== 모든 데이터 × 모든 기준 평가 ==========
# 평가 시 instruction에 RULE을 포함시킨다.
# → RULE 없이 생성된 데이터는 "말투 규칙 준수" 기준에서 낮은 점수를 받게 된다.
evaluation_instruction = GEN_SYSTEM_PROMPT + '\n' + RULE

evaluation_results = []
print('모든 합성 데이터 평가 시작...\n')

for i, data in enumerate(synthetic_data):
    rule_tag = '✅ RULE 적용' if data.get('_rule_applied') else '❌ RULE 미적용'
    print(f'[{i+1}] {data.get("movie_name", "Unknown")} ({rule_tag}) 평가 중...')
    data_output = json.dumps(data, ensure_ascii=False)

    scores = []
    comments = []
    for criteria in criteria_list:
        result = evaluate_with_llm(
            instruction=evaluation_instruction,
            output=data_output,
            criteria=criteria,
        )
        scores.append(result.get('score', 0))
        comments.append(result.get('comment', ''))

    avg_score = sum(scores) / len(scores) if scores else 0
    evaluation_results.append({
        'data': data,
        'scores': scores,
        'comments': comments,
        'avg': avg_score,
    })
    print(f'   점수: {scores}, 평균: {avg_score:.2f}')

# ========== 결과를 DataFrame으로 정리 ==========
criteria_labels = ['충실도', '구체성', '말투규칙']
rows = []
for r in evaluation_results:
    row = {
        'movie_name': r['data'].get('movie_name', ''),
        'genre': r['data'].get('_genre', ''),
        'RULE': '적용' if r['data'].get('_rule_applied') else '미적용',
    }
    for j, label in enumerate(criteria_labels):
        row[label] = r['scores'][j]
    row['평균'] = r['avg']
    rows.append(row)

df = pd.DataFrame(rows)
print('\n=== 평가 결과 요약 ===')
print(df.to_string(index=False))

# RULE 적용 vs 미적용 그룹별 평균 비교
print('\n=== RULE 적용 여부별 평균 점수 ===')
print(df.groupby('RULE')['평균'].mean().to_string())

print('\n→ RULE이 적용된 데이터는 "말투규칙" 기준에서 높은 점수를,')
print('  RULE이 미적용된 데이터는 낮은 점수를 받는 것을 확인할 수 있다.')
print('  이것이 LLM as Judge의 변별력이다.')


## 2-4. 마무리

### 오늘 만든 파이프라인

```
   ① 프롬프트 설계   ->   ② 합성 데이터 생성   ->   ③ LLM 채점   ->   ④ 선별
   Role + Few-shot        구조화 출력(JSON)        LLM as Judge      품질 필터
   + 조건 변주            4개 장르 x RULE 여부      3개 기준 평가      기준 미달 제거
```

이 흐름은 실제 AI 회사에서 학습 데이터를 준비할 때 그대로 쓰이는 구조다.
이후 챕터에서 배울 **파인튜닝의 재료**가 바로 이렇게 만들어진다.

### 핵심 정리 5가지

| # | 내용 |
|---|---|
| 1 | **합성은 무에서 유, 증강은 유에서 또 다른 유.** 데이터가 아예 없으면 합성이다 |
| 2 | **모호한 지시는 모호한 결과를 낳는다.** 역할·목표·조건을 구체적으로 |
| 3 | **형식은 부탁하는 게 아니라 강제한다.** JSON 스키마로 못 박는다 |
| 4 | **파라미터를 못 쓰면 프롬프트로 한다.** 다양성도, 일관성도 프롬프트로 확보 가능 |
| 5 | **LLM 채점은 1차 필터일 뿐이다.** 최종 판단과 표본 검수는 사람이 한다 |

---

## [참고 1] 데이터 증강 (Data Augmentation)

이번 실습에서는 LLM을 활용한 **데이터 합성(Synthesis)**, 즉 완전히 새로운 데이터를 생성하는 방법에 초점을 맞추었다.

관련 기법으로 **데이터 증강(Data Augmentation)** 이 있다:
- 기존 데이터를 기반으로 **변형** (이미지 회전, 텍스트 동의어 교체)을 가해 데이터 양을 늘리는 방식
- 모델의 강건성(Robustness)을 높이고 과적합을 방지하는 데 도움

**텍스트 데이터 증강 기법**

| 기법 | 방법 | 예시 |
|---|---|---|
| 동의어 교체 | 단어를 비슷한 뜻으로 바꿈 | "재미있다" -> "흥미롭다" |
| 역번역 (Back-translation) | 한국어 -> 영어 -> 한국어 | 의미는 같고 표현만 달라짐 |
| 랜덤 삽입·삭제·교체 | 단어를 무작위로 조작 | 노이즈에 강한 모델 학습 |
| **LLM 재작성** | "같은 뜻으로 다르게 써줘" | 가장 자연스럽지만 비용 발생 |

<br>

> **💡 한마디로 정리하면**
>
> - **합성**: 무(無)에서 유(有)를 창조 (새 질문-답변 쌍 생성)
> - **증강**: 유(有)에서 또 다른 유(有)를 생성 (기존 데이터 변형)
>
> 3-1 챕터의 `RandomResizedCrop`, `RandomHorizontalFlip` 등이 데이터 증강에 해당한다.

---

## [참고 2] 더 알아보기

**프롬프트 엔지니어링**
- 기법을 익히는 가장 좋은 방법은 **직접 바꿔가며 비교** 하는 것이다.
  오늘 실습 코드에서 프롬프트 한 줄씩 바꿔보고 결과가 어떻게 달라지는지 관찰해 보자.

**직접 해볼 만한 실험**

| 실험 | 방법 | 관찰할 것 |
|---|---|---|
| 장르 늘리기 | `genres` 리스트에 항목 추가 | 다양성이 실제로 늘어나는가 |
| RULE 바꾸기 | 말투 규칙을 다른 스타일로 | 채점 점수가 따라 움직이는가 |
| Few-shot 예시 수 | 예시를 1개 -> 5개로 | 형식 준수도가 계속 좋아지는가 |
| effort 비교 | 생성에 `low` / `high` 적용 | 품질 차이 vs 소요 시간 |
| 평가 기준 추가 | `criteria_list`에 기준 추가 | 어느 기준에서 점수가 낮은가 |
| 재평가 | 같은 셀을 두 번 실행 | 점수가 얼마나 흔들리는가 |
